# Prepare Digitized Panel-Rows for Zenodo and OpenStreetMap Upload

In GM-SEUS v1.0 contained 15,017 solar energy arrays (186 GWDC) covering 2,950 km² and included 2.92 million unique solar panel-rows (466 km²) within 9,042 of those arrays. This leaves 5,524 arrays without panel-row data due to outdated or lack of high-resolution imagery, poor quality delineation, and lack of existence in existing panel-row data sources. We ordered arrays with missing panel-rows by area, and digitized panel-rows using NAIP aerial imagery and satellite imagery where NAIP was not available. See our Workflow Documentation for more information. The OSM tagging appraoch was derived from the [OSM Solar Wiki](https://wiki.openstreetmap.org/wiki/Tag:generator:source%3Dsolar).

The output for Zenodo here is two GeoJSON objects: *GMSEUSv1_1_digAll.osm* and *GMSEUSv1_1_digUnique.osm*, with associated GeoJson objects.

## Import Libraries

In [44]:
# Import libraries
import pandas as pd
import geopandas as gpd
import os 
import sys

# Import packages needed for osm upload
from shapely.geometry import Polygon, MultiPolygon, LinearRing
from xml.etree.ElementTree import Element, SubElement, ElementTree
from datetime import datetime, timezone

# Import gmseusUtils - have to do it differently here because we are one layer deeper
base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(base_dir)
import gmseusUtils as gu
import importlib
gu = importlib.reload(gu)

## Set Required Paths and Variables

In [45]:
# Set folder paths
wd = gu.wd
downloaded_path = os.path.join(wd, r'Data\Downloaded')
derived_path = os.path.join(wd, r'Data\Derived')
derivedTemp_path = os.path.join(derived_path, r'intermediateProducts')

# Set an export folder path for GeoJSON and OSM files in prep for Zenodo
exportFolderPath = os.path.join(derivedTemp_path, r'GMSEUSv1_1_newlyDigitizedPanels')

# Set osm pull path
osmExistingPanelsPath = os.path.join(downloaded_path, r'SolarDB\OSM\OSMSolarPanels.shp')

# Set newly digitized panel path
newPanelsPath = os.path.join(derivedTemp_path, r'GMSEUSv1_1_newlyDigPanels.geojson')

# Set osm new dataset out paths: geojson and osm paths
osmDigGMSEUS_all_gjsonPath = os.path.join(exportFolderPath, r'GMSEUSv1_1_digAll.geojson')
osmDigGMSEUS_unique_gjsonPath = os.path.join(exportFolderPath, r'GMSEUSv1_1_digUnique.geojson')
osmDigGMSEUS_all_osmPath = os.path.join(exportFolderPath, r'GMSEUSv1_1_digAll.osm')
osmDigGMSEUS_unique_osmPath = os.path.join(exportFolderPath, r'GMSEUSv1_1_digUnique.osm')
osmDigArrays_all_gjsonPath = os.path.join(exportFolderPath, r'GMSEUSv1_1_digArraysAll.geojson')
osmDigArrays_unique_gjsonPath = os.path.join(exportFolderPath, r'GMSEUSv1_1_digArraysUnique.geojson')
osmDigArrays_all_osmPath = os.path.join(exportFolderPath, r'GMSEUSv1_1_digArraysAll.osm')
osmDigArrays_unique_osmPath = os.path.join(exportFolderPath, r'GMSEUSv1_1_digArraysUnique.osm')

# Set the primary OSM CRS for .osm and geojson files
osmToCRS = f'EPSG:4326' # Required by OSM also output of GEE so no reprojections required

# Load the config from the text file
config = gu.load_config(os.path.join(wd, r'Code\config.txt'))
minPanelRowArea = config['minPanelRowArea'] # 15 m2, minimum area for a single panel row from the 1st percentile panel area from Stid et al., 2022
maxPanelRowArea = config['maxPanelRowArea'] # 254 m2 95th perccentile for a single panel row from Stid et al., 2022. MSU Solar Carport has max 1890m2
minNumPanelRows = config['minNumPanelRows'] # 3 panels, minimum number of panels rows to form a ground mounted solar array, definition from Stid et al., 2022
minPmArRatio = config['minPmArRatio'] # 18.8%, 20% was minimum ratio of panel perimeter to area ratio for panels from Stid et al., 2022, MSU Solar Carport has min 18.9%
panelArrayBuff = config['panelArrayBuff'] # 10m buffer, 20m maximum distance between panel rows to form an array. We used 5m in Stid et al., 2022, but there are lower packing factors at greater latitudes (nativeID: '1229957948')
arrayArrayBuff = config['arrayArrayBuff'] # 20m buffer, 40m maximum distance between arrays subsections of the same mount type to form a complete array. In Stid et al., 2022, we used 50m, but we checked for same installation year in addition to mount type.
overlapDist = config['overlapDist'] # 190 meters, Set a overlap distance for checking if points/mismatched geometries between Solar PV datasets are duplicates
lengthRatioThresh = config['lengthRatioThresh']  # If length ratio < 3.0, set to dual_axis or else fixed_axis_diagonal, else single- or fixed-axis
areaRatioThresh = config['areaRatioThresh']  # If area ratio < 0.15, set to fixed_diag_axis, else dual_axis
toCRS = config['to_crs']  # EPSG:6350 NAD83 (2011)
toCRS = f'EPSG:{toCRS}'

# Check and create export folder if it doesn't exist
gu.checkFolder(exportFolderPath)

## Call Panel-Rows and Determine Conflation

### Get panel-row files

In [46]:
# Call new panels and ensure target CRS (do not use gmseusUtils)
newPanels = gpd.read_file(newPanelsPath).to_crs(toCRS)

# Call OSM existing panels and ensure target CRS
osmExistingPanels = gpd.read_file(osmExistingPanelsPath).to_crs(toCRS)

### Prepare newly digitized panel-row attributes

In [47]:
# Add Source column
newPanels['Source'] = 'GMSEUSv1_1_newlyDigPanels'

# Add a modType column and set to 'c-si'
newPanels['modType'] = 'c-si'

# Add an area column and a panelID column
newPanels['area'] = newPanels.geometry.area
newPanels['panelID'] = range(1, len(newPanels) + 1)

# Get the mount type, azimuth, length ratio, area ratio, short edge, and long edge for each panel
newPanels[['mount', 'azimuth', 'lengthRatio', 'shortEdge', 'longEdge']] = newPanels.apply(gu.assignMountType, axis=1, result_type='expand')
newPanels = newPanels.drop(columns=['lengthRatio', 'shortEdge', 'longEdge'])

### Create arrays from panel-rows and intersect with OSM panel-rows
Function from gmseusUtils is createArrayFromPanels(panels, buffDist, dissolveID, areaID='area')

In [48]:
# Create arrays from newPanels by a dummy column
newPanels['dissolveID'] = 1
newPanelsArrays = gu.createArrayFromPanels(newPanels, panelArrayBuff, 'dissolveID', 'area')
newPanels = newPanels.drop(columns=['dissolveID'])
newPanelsArrays = newPanelsArrays.explode(index_parts=False).reset_index(drop=True)
newPanelsArrays['Source'] = 'GMSEUSv1_1_newlyDigPanels' # Set array source for consistency with panels and export

# Create arrays from osmExistingPanels by a dummy column
osmExistingPanels['dissolveID'] = 1
osmPanelsArrays = gu.createArrayFromPanels(osmExistingPanels, panelArrayBuff, 'dissolveID', 'area')
osmExistingPanels = osmExistingPanels.drop(columns=['dissolveID'])
osmPanelsArrays = osmPanelsArrays.explode(index_parts=False).reset_index(drop=True)

# Add an array ID column to both geodataframes that is 1 through length of gdf
newPanelsArrays['arrayID'] = range(1, len(newPanelsArrays) + 1)
osmPanelsArrays['arrayID'] = range(1, len(osmPanelsArrays) + 1)

# Create a dummy column in osmPanelsArrays to join on
osmPanelsArrays['tempID'] = range(1, len(osmPanelsArrays) + 1)

# Perform left join between newPanelsArrays and osmPanelsArrays
osmNewjoined = gpd.sjoin(newPanelsArrays, osmPanelsArrays[['geometry', 'tempID']], how="left", predicate="intersects")

# Drop duplicate arrayID
osmNewjoined = osmNewjoined[~osmNewjoined['arrayID'].duplicated(keep='first')]

# Get conflating arrays, and check/drop duplicates
conflatingArraysDup = osmNewjoined[osmNewjoined['tempID'].notna()].copy()
conflating_arrayIDs = conflatingArraysDup['arrayID'].dropna().unique()
conflatingArrays = conflatingArraysDup[conflatingArraysDup['arrayID'].isin(conflating_arrayIDs)]

# Remove panel-rows and arrays that intersect with conflatingArrays
nonConflatingNewPanels = newPanels[~newPanels['geometry'].intersects(conflatingArrays['geometry'].unary_union)]
nonConflatingNewPanelsArrays = newPanelsArrays[~newPanelsArrays['geometry'].intersects(conflatingArrays['geometry'].unary_union)]

# Print how many arrays are in new panels, new panels non conflating, and OSM existing
print(f'There are {len(newPanels)} newly digitized panel-rows, {len(newPanelsArrays)} newly digitized arrays, and {len(osmPanelsArrays)} existing OSM arrays with panel-row data.')
print(f'There are {len(conflatingArrays)} conflated arrays with {len(newPanels) - len(nonConflatingNewPanels)} conflated panels')
print(f'There are {len(nonConflatingNewPanels)} non-conflating new panels to add to OSM.')
print(f'There are {len(nonConflatingNewPanelsArrays)} non-conflating arrays to add to OSM.')

# Print the proportion of nonConflatingNewPanels to osmExistingPanels added to OSM
propNonConflatingNewPanels = len(nonConflatingNewPanels) / len(osmExistingPanels) * 100
print(f'The proportion of newly digitized panel-rows to existing U.S. OSM panel-rows is {propNonConflatingNewPanels:.2f}%.')

# Print the number and proportion of non-conflating arraysadded to OSM
propArrayAddedOSM = (len(newPanelsArrays) - len(conflatingArrays)) / len(osmPanelsArrays) * 100
print(f'The proportion of newly digitized arrays to existing U.S. OSM arrays with panel-row data is {propArrayAddedOSM:.2f}%.')

There are 24182 newly digitized panel-rows, 1485 newly digitized arrays, and 6283 existing OSM arrays with panel-row data.
There are 35 conflated arrays with 312 conflated panels
There are 23870 non-conflating new panels to add to OSM.
There are 1450 non-conflating arrays to add to OSM.
The proportion of newly digitized panel-rows to existing U.S. OSM panel-rows is 2.11%.
The proportion of newly digitized arrays to existing U.S. OSM arrays with panel-row data is 23.08%.


## Add OSM Attributes and Prepare for Export

In [49]:
# List of keys we want to export as tags
tag_keys = [
    "power",
    "generator:source",
    "generator:method",
    "generator:output:electricity",
    "generator:type",
    "generator:solar:tracking",
    "location"]

# Function to add required OSM tags to a GeoDataFrame
def add_osm_tags(gdf, mount_col="mount"):
    g = gdf.copy()
    g["power"] = "generator"
    g["generator:source"] = "solar"
    g["generator:method"] = "photovoltaic"
    g["generator:output:electricity"] = "yes"
    g["generator:type"] = "solar_photovoltaic_panel"
    # Use the mount column to determine tracking: "fixed_axis" = no, otherwise yes
    g["generator:solar:tracking"] = g.get(mount_col, None).apply(lambda v: "no" if isinstance(v, str) and v.lower() == "fixed_axis" else "yes")
    g["location"] = "surface"
    g["direction"] = g.get("azimuth", None)
    return g

# Function to export a GeoDataFrame as an OSM XML file
def write_osm(gdf, out_path, ref_col=None):

    # Create OSM XML root
    osm = Element("osm", attrib={"version": "0.6", "generator": "gmseus-osm-writer"})
    ts = datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

    # OSM requires negative IDs for new objects
    next_node_id, next_way_id = -1, -1

    # Helper to add a node element to the OSM file
    def add_node(lon, lat):
        nonlocal next_node_id
        SubElement(osm, "node", {
            "id": str(next_node_id),
            "lat": f"{lat:.7f}",
            "lon": f"{lon:.7f}",
            "version": "1",
            "timestamp": ts,
        })
        nid = next_node_id
        next_node_id -= 1
        return nid

    # Helper to add a way element (closed polygon) to the OSM file
    def add_way(node_ids, tags):
        nonlocal next_way_id
        way = SubElement(osm, "way", {
            "id": str(next_way_id),
            "version": "1",
            "timestamp": ts,
        })
        for nid in node_ids:
            SubElement(way, "nd", {"ref": str(nid)})
        for k, v in tags.items():
            if v is not None and v != "":
                SubElement(way, "tag", {"k": k, "v": str(v)})
        next_way_id -= 1

    # Ensure polygon rings are closed
    def outer_coords(poly):
        ring = LinearRing(poly.exterior.coords)
        coords = list(ring.coords)
        if coords[0] != coords[-1]:
            coords.append(coords[0])
        return coords

    # Iterate through all features in the GeoDataFrame
    for _, row in gdf.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue

        # Build tags for this feature
        tags = {k: (row[k] if k in row else None) for k in tag_keys}
        if ref_col and ref_col in row and pd.notna(row[ref_col]):
            tags["ref"] = str(row[ref_col])

        # Handle polygons and multipolygons
        parts = geom.geoms if isinstance(geom, MultiPolygon) else [geom]
        for part in parts:
            if not isinstance(part, Polygon):
                continue
            if not part.is_valid:
                part = part.buffer(0)
                if part.is_empty:
                    continue

            # Convert outer ring into OSM nodes
            coords = outer_coords(part)
            node_ids = [add_node(lon, lat) for lon, lat in coords]

            # Ensure the first and last node are the same (closed polygon)
            if node_ids[0] != node_ids[-1]:
                node_ids.append(node_ids[0])

            # Add the way element to the OSM file
            add_way(node_ids, tags)

    # Write the OSM XML tree to file
    ElementTree(osm).write(out_path, encoding="utf-8", xml_declaration=True)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Prepare for GeoJSON and OSM Export

# Transform all gdf to osmToCRS
newPanels = newPanels.to_crs(osmToCRS)
newPanelsArrays = newPanelsArrays.to_crs(osmToCRS)
nonConflatingNewPanels = nonConflatingNewPanels.to_crs(osmToCRS)
nonConflatingNewPanelsArrays = nonConflatingNewPanelsArrays.to_crs(osmToCRS)

# Apply OSM tags to both GeoDataFrames
newPanels_tagged = add_osm_tags(newPanels, mount_col="mount")
nonConflatingNewPanels_tagged = add_osm_tags(nonConflatingNewPanels, mount_col="mount")

# Drop all columns from panel-rows except for OSM tags, geometry, Source, and panelID. For repository purposes, save the GeoJSONs with panel IDs and Source
newPanels_tagged = newPanels_tagged[tag_keys + ['geometry', 'Source', 'panelID']]
nonConflatingNewPanels_tagged = nonConflatingNewPanels_tagged[tag_keys + ['geometry', 'Source', 'panelID']]

# Drop all columns from arrays except for Source, arrayID, and geometry
newPanelsArrays = newPanelsArrays[['Source', 'arrayID', 'geometry']]
nonConflatingNewPanelsArrays = nonConflatingNewPanelsArrays[['Source', 'arrayID', 'geometry']]

# Export to GeoJSON
newPanels_tagged.to_file(osmDigGMSEUS_all_gjsonPath, driver="GeoJSON")
nonConflatingNewPanels_tagged.to_file(osmDigGMSEUS_unique_gjsonPath, driver="GeoJSON")
newPanelsArrays.to_file(osmDigArrays_all_gjsonPath, driver="GeoJSON")
nonConflatingNewPanelsArrays.to_file(osmDigArrays_unique_gjsonPath, driver="GeoJSON")

# For .osm purposes, drop Source and panelID
newPanels_tagged = newPanels_tagged.drop(columns=['Source', 'panelID'])
nonConflatingNewPanels_tagged = nonConflatingNewPanels_tagged.drop(columns=['Source', 'panelID'])

# Export to OSM XML
write_osm(newPanels_tagged, osmDigGMSEUS_all_osmPath, ref_col=None)
write_osm(nonConflatingNewPanels_tagged, osmDigGMSEUS_unique_osmPath, ref_col=None)
write_osm(newPanelsArrays, osmDigArrays_all_osmPath, ref_col=None)
write_osm(nonConflatingNewPanelsArrays, osmDigArrays_unique_osmPath, ref_col=None)